# Semantic Energy — Colab Backend Deployment

This notebook deploys the **Semantic Energy** backend on Google Colab with a public URL via **ngrok**.

**Architecture:** Vercel (frontend) + Colab (backend) + ngrok (tunnel)

**What you get:**
- Full FastAPI backend with all endpoints (Full SE, Fast SLT, Fast TBG)
- Trained hallucination probes loaded from HuggingFace Hub
- Model switching between Llama 3.1 8B and Qwen3 8B
- Public URL via ngrok static domain

---

## Prerequisites

1. **Runtime → Change runtime type → T4 GPU**
2. Get a **free ngrok auth token** from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)
3. (Optional) Get your **ngrok static domain** from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains)

> ⚠️ **Important:** Make sure you've selected a **T4 GPU runtime** before running the cells below.

## Step 1: Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

## Step 2: Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn pyngrok scikit-learn spacy pysbd huggingface_hub
!python -m spacy download en_core_web_sm -q
print("Dependencies installed!")

## Step 3: Clone the Repository

In [ ]:
import os

REPO_URL = "https://github.com/DangerDudeSL/SemanticEnergyV1.git"
REPO_DIR = "SemanticEnergyV1"

if os.path.exists(REPO_DIR):
    print(f"{REPO_DIR} already exists, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL}
    print(f"Cloned {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 4: Configuration

Set your ngrok token and choose which model to load.

**ngrok static domain:** Free accounts get 1 permanent domain from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains). Using a static domain means your URL never changes between sessions.

In [ ]:
# ===== CONFIGURATION =====

# ngrok auth token (get from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = ""  # <-- Paste your token here

# ngrok static domain (get from https://dashboard.ngrok.com/domains)
# Leave empty to get a random URL each session
NGROK_STATIC_DOMAIN = ""  # e.g. "your-domain.ngrok-free.dev"

# Model to load on startup
# Options: "meta-llama/Llama-3.1-8B-Instruct" or "Qwen/Qwen3-8B"
DEFAULT_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# HuggingFace token (needed for gated models like Llama)
HF_TOKEN = ""  # <-- Paste your HF token here if using Llama

# HuggingFace repo for probe bundles
PROBE_HF_REPO = "DangerDudeSL/semantic-energy-probes"

# =========================

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Default model: {DEFAULT_MODEL}")
print(f"ngrok static domain: {NGROK_STATIC_DOMAIN or '(random)'}")
print(f"Probe repo: {PROBE_HF_REPO}")

## Step 5: Download Probe Bundles

Downloads trained probe pkl files from HuggingFace Hub into `backend/models/`.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

MODELS_DIR = os.path.join(os.getcwd(), "backend", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Probe bundles to download
PROBE_FILES = {
    "meta-llama/Llama-3.1-8B-Instruct": "probes_llama3-8b_triviaqa.pkl",
    "Qwen/Qwen3-8B": "probes_qwen3-8b_triviaqa.pkl",
}

for model_id, probe_file in PROBE_FILES.items():
    dest = os.path.join(MODELS_DIR, probe_file)
    if os.path.exists(dest):
        print(f"  {probe_file} already exists, skipping.")
        continue
    try:
        downloaded = hf_hub_download(
            repo_id=PROBE_HF_REPO,
            filename=probe_file,
        )
        shutil.copy2(downloaded, dest)
        print(f"  Downloaded {probe_file}")
    except Exception as e:
        print(f"  Could not download {probe_file}: {e}")
        print(f"  (Probes for {model_id} will be unavailable)")

print(f"\nProbe bundles in {MODELS_DIR}:")
for f in os.listdir(MODELS_DIR):
    size_mb = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

## Step 6: Set Up ngrok

In [ ]:
from pyngrok import ngrok

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok authenticated!")

## Step 7: Load Model & Start Server

This cell:
1. Loads the LLM into GPU memory
2. Loads the matching probe bundle
3. Starts the FastAPI server with all endpoints
4. Creates the ngrok tunnel

> First run takes **2-5 minutes** (downloading model weights). Subsequent runs use cached weights.

In [ ]:
import sys
import gc
import pickle
import threading
import traceback
import torch
import torch.nn.functional as F

# Add backend to path
sys.path.insert(0, os.path.join(os.getcwd(), "backend"))
from engine import SemanticEngine, cal_flow, sum_normalize

# ── Load model ──────────────────────────────────────────────────────────────
print(f"\nLoading model: {DEFAULT_MODEL}", flush=True)
engine = SemanticEngine(model_id=DEFAULT_MODEL)
current_model_id = DEFAULT_MODEL
loading_model_id = None
print(f"Model loaded on {engine.device}!", flush=True)

if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used: {mem_used:.1f} GB")

# ── Load probe bundle ────────────────────────────────────────────────────────
PROBE_BUNDLES = {
    "meta-llama/Llama-3.1-8B-Instruct": "probes_llama3-8b_triviaqa.pkl",
    "Qwen/Qwen3-8B": "probes_qwen3-8b_triviaqa.pkl",
}

def _patch_sklearn_compat(bundle):
    """Fix probes trained with scikit-learn <1.6 running on >=1.6 (multi_class removed)."""
    for key in bundle:
        obj = bundle[key]
        if hasattr(obj, 'predict_proba') and hasattr(obj, 'multi_class'):
            delattr(obj, 'multi_class')

def load_probe_bundle(model_id):
    filename = PROBE_BUNDLES.get(model_id)
    if not filename:
        print(f"No probe bundle configured for {model_id}")
        return None
    path = os.path.join(MODELS_DIR, filename)
    if not os.path.exists(path):
        print(f"Probe bundle not found at {path}")
        return None
    with open(path, "rb") as f:
        bundle = pickle.load(f)
    _patch_sklearn_compat(bundle)
    print(f"Probe bundle loaded: {filename}")
    return bundle

probe_bundle = load_probe_bundle(current_model_id)

In [ ]:
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uvicorn
import numpy as np

app = FastAPI(title="Semantic Energy API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── /status ──────────────────────────────────────────────────────────────────
@app.get("/status")
async def status_endpoint():
    if engine is not None and loading_model_id is None:
        return {"ready": True, "model_id": current_model_id}
    return {
        "ready": False,
        "loading_model_id": loading_model_id or current_model_id or "unknown",
    }

# ── /switch_model ────────────────────────────────────────────────────────────
@app.post("/switch_model")
async def switch_model_endpoint(request: Request):
    global engine, current_model_id, loading_model_id, probe_bundle
    try:
        data = await request.json()
        requested_model = data.get("model_id", "")
        if not requested_model:
            return JSONResponse({"error": "model_id is required"}, status_code=400)
        if requested_model == current_model_id and engine is not None:
            return {"status": "already_loaded", "model_id": current_model_id}

        loading_model_id = requested_model
        print(f"\nModel switch: {current_model_id} -> {requested_model}", flush=True)

        if engine is not None:
            print("Unloading current model...", flush=True)
            del engine.model
            del engine.tokenizer
            del engine
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        print(f"Loading: {requested_model}...", flush=True)
        current_model_id = requested_model
        engine = SemanticEngine(model_id=current_model_id)
        loading_model_id = None
        probe_bundle = load_probe_bundle(current_model_id)
        print(f"Model {current_model_id} is READY!", flush=True)
        return {"status": "loaded", "model_id": current_model_id}

    except Exception as e:
        loading_model_id = None
        traceback.print_exc()
        return JSONResponse({"error": str(e)}, status_code=500)

# ── /chat (Full Semantic Energy) ─────────────────────────────────────────────
@app.post("/chat")
async def chat_endpoint(request: Request):
    global engine, current_model_id, loading_model_id, probe_bundle
    if engine is None:
        return JSONResponse({"error": "Model is still loading..."}, status_code=503)
    try:
        data = await request.json()
        user_prompt = data.get("prompt", "")
        num_samples = data.get("num_samples", 5)
        requested_model = data.get("model_id", current_model_id)

        # Dynamic model switching
        if requested_model != current_model_id:
            loading_model_id = requested_model
            if engine is not None:
                del engine.model; del engine.tokenizer; del engine
                gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
            current_model_id = requested_model
            engine = SemanticEngine(model_id=current_model_id)
            loading_model_id = None
            probe_bundle = load_probe_bundle(current_model_id)

        if not user_prompt:
            return JSONResponse({"error": "Prompt is required"}, status_code=400)

        print(f"\n[1/3] Generating {num_samples} responses for: {user_prompt}", flush=True)
        generated_data = engine.generate_responses(user_prompt, num_samples=num_samples)
        main_answer = generated_data[0]["answer"]

        # B1 sentence-level logit confidence
        sentence_scores = engine.score_sentences(
            main_answer, generated_data[0]["token_ids"],
            generated_data[0]["logits"], generated_data[0].get("top2_logits"),
        )

        print(f"[2/3] Clustering...", flush=True)
        answer_texts = [d["answer"] for d in generated_data]
        clusters = engine.find_semantic_clusters(user_prompt, answer_texts)
        print(f"Found {len(clusters)} clusters: {clusters}", flush=True)

        print(f"[3/3] Computing Semantic Energy...", flush=True)
        probs_list = [d['probs'] for d in generated_data]
        logits_list = [d['logits'] for d in generated_data]
        probs_se, logits_se = cal_flow(probs_list, logits_list, clusters, fermi_mu=None)

        main_cluster_idx = 0
        for idx, cluster in enumerate(clusters):
            if 0 in cluster:
                main_cluster_idx = idx
                break

        cluster_energies = sum_normalize(logits_se)
        main_confidence = cluster_energies[main_cluster_idx]

        if main_confidence > 0.80:
            confidence_level = "high"
        elif main_confidence > 0.50:
            confidence_level = "medium"
        else:
            confidence_level = "low"

        valid_confs = [s["confidence"] for s in sentence_scores
                       if s["confidence"] is not None and s.get("is_claim", True)]
        sentence_avg_confidence = float(sum(valid_confs) / len(valid_confs)) if valid_confs else None

        print(f"Confidence: {main_confidence:.2f} ({confidence_level})", flush=True)

        return {
            "answer": main_answer,
            "confidence_score": main_confidence,
            "confidence_level": confidence_level,
            "clusters_found": len(clusters),
            "sentence_scores": sentence_scores,
            "sentence_avg_confidence": round(sentence_avg_confidence, 4) if sentence_avg_confidence is not None else None,
            "debug_data": {
                "all_answers": answer_texts,
                "energies_per_cluster": cluster_energies,
            },
        }
    except Exception as e:
        traceback.print_exc()
        return JSONResponse({"error": str(e)}, status_code=500)

# ── /score_fast_tbg (Pre-generation probe) ───────────────────────────────────
@app.post("/score_fast_tbg")
async def score_fast_tbg(request: Request):
    if engine is None:
        return JSONResponse({"error": "Model is still loading..."}, status_code=503)
    if probe_bundle is None:
        return JSONResponse({"error": "No probe bundle loaded for this model."}, status_code=503)
    try:
        data = await request.json()
        prompt = data.get("prompt", "")
        if not prompt:
            return JSONResponse({"error": "Prompt is required"}, status_code=400)
        return engine.score_with_tbg_probe(prompt, probe_bundle)
    except Exception as e:
        traceback.print_exc()
        return JSONResponse({"error": str(e)}, status_code=500)

# ── /score_fast_slt (Post-generation probe) ──────────────────────────────────
@app.post("/score_fast_slt")
async def score_fast_slt(request: Request):
    if engine is None:
        return JSONResponse({"error": "Model is still loading..."}, status_code=503)
    if probe_bundle is None:
        return JSONResponse({"error": "No probe bundle loaded for this model."}, status_code=503)
    try:
        data = await request.json()
        prompt = data.get("prompt", "")
        if not prompt:
            return JSONResponse({"error": "Prompt is required"}, status_code=400)
        return engine.score_with_slt_probe(prompt, probe_bundle)
    except Exception as e:
        traceback.print_exc()
        return JSONResponse({"error": str(e)}, status_code=500)

# ── Start server + ngrok tunnel ──────────────────────────────────────────────
PORT = 8000

if NGROK_STATIC_DOMAIN:
    public_url = ngrok.connect(PORT, domain=NGROK_STATIC_DOMAIN)
else:
    public_url = ngrok.connect(PORT)

print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"{'='*60}")
print(f"\nSet this as BACKEND_URL in your Vercel frontend's script.js")
print(f"Or open it directly if serving frontend from same origin.")
print(f"\nEndpoints available:")
print(f"  GET  /status")
print(f"  POST /switch_model")
print(f"  POST /chat")
print(f"  POST /score_fast_tbg")
print(f"  POST /score_fast_slt")

# Run in background thread so cell doesn't block
threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": PORT, "log_level": "warning"},
    daemon=True,
).start()

print("\nServer is running! The URL stays active as long as this Colab session is running.")

## Step 8: Quick Test

Verify all endpoints are working.

In [ ]:
import requests
import json
import time

BASE = f"http://localhost:{PORT}"
time.sleep(2)  # Wait for server thread to start

# Test /status
print("Testing /status...")
r = requests.get(f"{BASE}/status")
print(f"  {r.json()}\n")

# Test /score_fast_tbg (fastest — no generation)
print("Testing /score_fast_tbg...")
r = requests.post(f"{BASE}/score_fast_tbg", json={"prompt": "What is the capital of France?"})
if r.status_code == 200:
    d = r.json()
    print(f"  Energy risk: {d['energy_risk']:.3f}")
    print(f"  Entropy risk: {d['entropy_risk']:.3f}")
    print(f"  Combined risk: {d['combined_risk']:.3f}")
    print(f"  Confidence: {d['confidence_level']}\n")
else:
    print(f"  Error {r.status_code}: {r.text}\n")

# Test /score_fast_slt (generates 1 answer + probe scoring)
print("Testing /score_fast_slt...")
r = requests.post(f"{BASE}/score_fast_slt", json={"prompt": "What is the capital of France?"})
if r.status_code == 200:
    d = r.json()
    print(f"  Answer: {d['answer'][:80]}...")
    print(f"  Combined risk: {d['combined_risk']:.3f}")
    print(f"  Confidence: {d['confidence_level']}")
    print(f"  Sentences: {len(d.get('sentence_scores', []))}\n")
else:
    print(f"  Error {r.status_code}: {r.text}\n")

print("All endpoints verified!")

---

## Notes

- **Session duration:** Free Colab sessions last ~12 hours max. If using a static ngrok domain, the URL stays the same when you restart.
- **Model caching:** After the first download, model weights are cached in `/root/.cache/huggingface` until the runtime resets.
- **GPU memory:** Check usage with `!nvidia-smi` in a new cell.
- **Switching models at runtime:** Use the dropdown in the Vercel frontend, or POST to `/switch_model` with `{"model_id": "Qwen/Qwen3-8B"}`.
- **Frontend deployment:** Deploy `frontend/` to Vercel with root directory set to `frontend/`. Set `BACKEND_URL` in `script.js` to your ngrok domain.

---

*Powered by [Semantic Energy](https://arxiv.org/abs/2508.14496) — Detecting LLM Hallucination Beyond Entropy*